In [1]:
from systems import build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

from dynamiqs.steady_state.solvers.steady_state_arnoldi2 import SteadyStateArnoldi, SteadyStateArnoldiResult

jax.config.update("jax_enable_x64", True)
n = 40
nd = 3
deltas = jnp.linspace(0, 5, nd) * 2 * jnp.pi
deltas = [3 * 2 * jnp.pi]


def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


def print_top10_spectrum(label, eigvals):
    idx = jnp.argsort(-jnp.abs(eigvals))[:10]
    print(label)
    for rank, i in enumerate(idx, start=1):
        lam = complex(eigvals[i])
        print(
            f"  {rank:2d}: {lam.real:+.6e}{lam.imag:+.6e}j  |λ|={abs(lam):.6e}"
        )


for delta in deltas:
    H, Ls = build_kerr_oscillator(n, delta)
    dims = H.dims
    dtype = H.to_jax().dtype
    dim = n * n

    Ls_q = dq.stack(Ls)
    H_jax = H.to_jax()
    LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
    G = -1j * H_jax - 0.5 * LdagL
    lyap = LyapunovSolverEig(G)

    def K_vec(x):
        rho = to_dm(x, n=n, dims=dims)
        return from_matrix((Ls_q @ rho @ Ls_q.dag()).sum(0).to_jax())

    def S_inv_vec(x):
        return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

    K_mat = build_operator_matrix(K_vec, dim, dtype)
    S_inv_mat = build_operator_matrix(S_inv_vec, dim, dtype)

    S_inv_K = S_inv_mat @ K_mat
    K_S_inv = K_mat @ S_inv_mat

    eigvals_S_inv_K, eigvecs_S_inv_K = jnp.linalg.eig(S_inv_K)
    eigvals_K_S_inv, eigvecs_K_S_inv = jnp.linalg.eig(K_S_inv)

    print_top10_spectrum("S_inv @ K spectrum:", eigvals_S_inv_K)
    print_top10_spectrum("K @ S_inv spectrum:", eigvals_K_S_inv)

    idx = jnp.argsort(-jnp.abs(eigvals_S_inv_K))[0]
    eigvec_S_inv_K = eigvecs_S_inv_K[:, idx]
    norminf = jnp.max(
        jnp.abs(
            dq.lindbladian(
                H, Ls, to_dm(eigvec_S_inv_K, n=n, dims=dims)
            ).to_jax()
        )
    )
    print(f"max|L(rho)| = {float(norminf):.6e}")

S_inv @ K spectrum:
   1: -1.000004e+00+2.835734e-07j  |λ|=1.000004e+00
   2: -9.279175e-01+8.445401e-02j  |λ|=9.317529e-01
   3: -9.279165e-01-8.445553e-02j  |λ|=9.317520e-01
   4: -8.732741e-01+2.864022e-06j  |λ|=8.732741e-01
   5: -8.406631e-01+1.603244e-01j  |λ|=8.558144e-01
   6: -8.406620e-01-1.603171e-01j  |λ|=8.558120e-01
   7: -8.011363e-01+7.980030e-02j  |λ|=8.051009e-01
   8: -8.011361e-01-7.978963e-02j  |λ|=8.050996e-01
   9: -7.448771e-01+2.239498e-01j  |λ|=7.778145e-01
  10: -7.448809e-01-2.239362e-01j  |λ|=7.778143e-01
K @ S_inv spectrum:
   1: -1.000004e+00+2.835734e-07j  |λ|=1.000004e+00
   2: -9.279175e-01+8.445401e-02j  |λ|=9.317529e-01
   3: -9.279165e-01-8.445553e-02j  |λ|=9.317520e-01
   4: -8.732741e-01+2.864022e-06j  |λ|=8.732741e-01
   5: -8.406631e-01+1.603244e-01j  |λ|=8.558144e-01
   6: -8.406620e-01-1.603171e-01j  |λ|=8.558120e-01
   7: -8.011363e-01+7.980030e-02j  |λ|=8.051009e-01
   8: -8.011361e-01-7.978963e-02j  |λ|=8.050996e-01
   9: -7.448771e-01+2.23

In [1]:
from systems import build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

jax.config.update("jax_enable_x64", True)
n = 40
nd = 1
deltas = jnp.linspace(0.1, 5, nd) * 2 * jnp.pi
deltas= [3 * 2 * jnp.pi]

def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


def compute_norminf(eigvecs, eigvals, H, Ls, n, dims):
    idx = jnp.argsort(-jnp.abs(eigvals))[0]
    eigvec = eigvecs[:, idx]
    rho = to_dm(eigvec, n=n, dims=dims)
    return float(jnp.max(jnp.abs(dq.lindbladian(H, Ls, rho).to_jax())))


norminf_sinvk_lyap = []
norminf_ksinv_lyap = []
norminf_sinvk_direct = []
norminf_ksinv_direct = []

for i, delta in enumerate(deltas):
    print(f"[{i+1}/{nd}] delta/(2pi) = {float(delta / (2*jnp.pi)):.2f}")

    H, Ls = build_kerr_oscillator(n, delta)
    dims = H.dims
    dtype = H.to_jax().dtype
    dim = n * n

    Ls_q = dq.stack(Ls)
    H_jax = H.to_jax()
    LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
    G = -1j * H_jax - 0.5 * LdagL

    lyap = LyapunovSolverEig(G)

    # --- K matrix via dq.spre / dq.spost ---
    # K = sum_k L_k . rho . L_k^dag  =>  sum_k spre(L_k) @ spost(L_k^dag)
    K_mat = jnp.zeros((dim, dim), dtype=dtype)
    for L in Ls:
        K_mat = K_mat + dq.sprepost(L, L.dag()).to_jax()

    # --- S matrix via kron / spre / spost ---
    # S(rho) = G rho + rho G^dag  =>  spre(G) + spost(G^dag)
    G_qa = dq.asqarray(G, dims=dims)
    S_mat = (dq.spre(G_qa) + dq.spost(G_qa.dag())).to_jax()

    # --- S_inv direct ---
    S_inv_mat_direct = jnp.linalg.inv(S_mat)

    # --- S_inv via Lyapunov ---
    def S_inv_vec_lyap(x):
        return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

    S_inv_mat_lyap = build_operator_matrix(S_inv_vec_lyap, dim, dtype)

    # --- Compose ---
    S_inv_K_lyap = S_inv_mat_lyap @ K_mat
    S_inv_K_direct = S_inv_mat_direct @ K_mat
    K_S_inv_lyap = K_mat @ S_inv_mat_lyap
    K_S_inv_direct = K_mat @ S_inv_mat_direct

    # --- Eigendecompositions ---
    evals_sinvk_lyap, evecs_sinvk_lyap = jnp.linalg.eig(S_inv_K_lyap)
    evals_sinvk_direct, evecs_sinvk_direct = jnp.linalg.eig(S_inv_K_direct)
    evals_ksinv_lyap, evecs_ksinv_lyap = jnp.linalg.eig(K_S_inv_lyap)
    evals_ksinv_direct, evecs_ksinv_direct = jnp.linalg.eig(K_S_inv_direct)

    # --- Norm inf ---
    norminf_sinvk_lyap.append(
        compute_norminf(evecs_sinvk_lyap, evals_sinvk_lyap, H, Ls, n, dims)
    )
    norminf_sinvk_direct.append(
        compute_norminf(evecs_sinvk_direct, evals_sinvk_direct, H, Ls, n, dims)
    )
    norminf_ksinv_lyap.append(
        compute_norminf(evecs_ksinv_lyap, evals_ksinv_lyap, H, Ls, n, dims)
    )
    norminf_ksinv_direct.append(
        compute_norminf(evecs_ksinv_direct, evals_ksinv_direct, H, Ls, n, dims)
    )

# --- Plot ---
print("norminf_sinvk_lyap:", norminf_sinvk_lyap)
print("norminf_sinvk_direct:", norminf_sinvk_direct)

print("norminf_ksinv_lyap:", norminf_ksinv_lyap)
print("norminf_ksinv_direct:", norminf_ksinv_direct)

[1/1] delta/(2pi) = 3.00
norminf_sinvk_lyap: [0.001092797424851868]
norminf_sinvk_direct: [7.350930813807945e-11]
norminf_ksinv_lyap: [1.3845014488798197]
norminf_ksinv_direct: [1.384147187003196]


In [4]:
from systems import build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

jax.config.update("jax_enable_x64", True)
n = 40
nd = 1
deltas = jnp.linspace(0.1, 5, nd) * 2 * jnp.pi
deltas= [3 * 2 * jnp.pi]

def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


def compute_norminf(eigvecs, eigvals, H, Ls, n, dims):
    idx = jnp.argsort(-jnp.abs(eigvals))[0]
    eigvec = eigvecs[:, idx]
    rho = to_dm(eigvec, n=n, dims=dims)
    return float(jnp.max(jnp.abs(dq.lindbladian(H, Ls, rho).to_jax())))


norminf_sinvk_lyap = []
norminf_ksinv_lyap = []
norminf_sinvk_direct = []
norminf_ksinv_direct = []

for i, delta in enumerate(deltas):
    print(f"[{i+1}/{nd}] delta/(2pi) = {float(delta / (2*jnp.pi)):.2f}")

    H, Ls = build_kerr_oscillator(n, delta)
    dims = H.dims
    dtype = H.to_jax().dtype
    dim = n * n

    Ls_q = dq.stack(Ls)
    H_jax = H.to_jax()
    LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
    G = -1j * H_jax - 0.5 * LdagL

    lyap = LyapunovSolverEig(G,n_refinement=3)

    # --- K matrix via dq.spre / dq.spost ---
    # K = sum_k L_k . rho . L_k^dag  =>  sum_k spre(L_k) @ spost(L_k^dag)
    K_mat = jnp.zeros((dim, dim), dtype=dtype)
    for L in Ls:
        K_mat = K_mat + dq.sprepost(L, L.dag()).to_jax()

    # --- S matrix via kron / spre / spost ---
    # S(rho) = G rho + rho G^dag  =>  spre(G) + spost(G^dag)
    G_qa = dq.asqarray(G, dims=dims)
    S_mat = (dq.spre(G_qa) + dq.spost(G_qa.dag())).to_jax()

    # --- S_inv direct ---
    S_inv_mat_direct = jnp.linalg.inv(S_mat)

    # --- S_inv via Lyapunov ---
    def S_inv_vec_lyap(x):
        return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

    S_inv_mat_lyap = build_operator_matrix(S_inv_vec_lyap, dim, dtype)

    # --- Compose ---
    S_inv_K_lyap = S_inv_mat_lyap @ K_mat
    S_inv_K_direct = S_inv_mat_direct @ K_mat
    K_S_inv_lyap = K_mat @ S_inv_mat_lyap
    K_S_inv_direct = K_mat @ S_inv_mat_direct

    # --- Eigendecompositions ---
    evals_sinvk_lyap, evecs_sinvk_lyap = jnp.linalg.eig(S_inv_K_lyap)
    evals_sinvk_direct, evecs_sinvk_direct = jnp.linalg.eig(S_inv_K_direct)
    evals_ksinv_lyap, evecs_ksinv_lyap = jnp.linalg.eig(K_S_inv_lyap)
    evals_ksinv_direct, evecs_ksinv_direct = jnp.linalg.eig(K_S_inv_direct)

    # --- Norm inf ---
    norminf_sinvk_lyap.append(
        compute_norminf(evecs_sinvk_lyap, evals_sinvk_lyap, H, Ls, n, dims)
    )
    norminf_sinvk_direct.append(
        compute_norminf(evecs_sinvk_direct, evals_sinvk_direct, H, Ls, n, dims)
    )
    norminf_ksinv_lyap.append(
        compute_norminf(evecs_ksinv_lyap, evals_ksinv_lyap, H, Ls, n, dims)
    )
    norminf_ksinv_direct.append(
        compute_norminf(evecs_ksinv_direct, evals_ksinv_direct, H, Ls, n, dims)
    )

# --- Plot ---
print("norminf_sinvk_lyap:", norminf_sinvk_lyap)
print("norminf_sinvk_direct:", norminf_sinvk_direct)

print("norminf_ksinv_lyap:", norminf_ksinv_lyap)
print("norminf_ksinv_direct:", norminf_ksinv_direct)

[1/1] delta/(2pi) = 3.00
norminf_sinvk_lyap: [6.182504708399003e-11]
norminf_sinvk_direct: [7.350930813807945e-11]
norminf_ksinv_lyap: [1.3841471870032023]
norminf_ksinv_direct: [1.384147187003196]


In [1]:
from systems import build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

jax.config.update("jax_enable_x64", True)
n = 40
nd = 1
deltas = jnp.linspace(0.1, 5, nd) * 2 * jnp.pi
deltas = [3 * 2 * jnp.pi]


def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


def compute_norminf(eigvecs, eigvals, H, Ls, n, dims):
    idx = jnp.argsort(-jnp.abs(eigvals))[0]
    eigvec = eigvecs[:, idx]
    rho = to_dm(eigvec, n=n, dims=dims)
    return float(jnp.max(jnp.abs(dq.lindbladian(H, Ls, rho).to_jax())))


def get_dominant_eigen(eigvals, eigvecs):
    """Return (dominant eigenvalue, associated eigenvector)."""
    idx = jnp.argsort(-jnp.abs(eigvals))[0]
    return eigvals[idx], eigvecs[:, idx]


def overlap(v1, v2):
    """Compute |<v1|v2>| / (||v1|| ||v2||)."""
    return float(jnp.abs(jnp.vdot(v1, v2)) / (jnp.linalg.norm(v1) * jnp.linalg.norm(v2)))


norminf_sinvk_lyap = []
norminf_ksinv_lyap = []
norminf_sinvk_direct = []
norminf_ksinv_direct = []

for i, delta in enumerate(deltas):
    print(f"[{i+1}/{nd}] delta/(2pi) = {float(delta / (2*jnp.pi)):.2f}")

    H, Ls = build_kerr_oscillator(n, delta)
    dims = H.dims
    dtype = H.to_jax().dtype
    dim = n * n

    Ls_q = dq.stack(Ls)
    H_jax = H.to_jax()
    LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
    G = -1j * H_jax - 0.5 * LdagL

    lyap = LyapunovSolverEig(G, n_refinement=3)

    # --- K matrix ---
    K_mat = jnp.zeros((dim, dim), dtype=dtype)
    for L in Ls:
        K_mat = K_mat + dq.sprepost(L, L.dag()).to_jax()

    # --- S matrix ---
    G_qa = dq.asqarray(G, dims=dims)
    S_mat = (dq.spre(G_qa) + dq.spost(G_qa.dag())).to_jax()

    # --- S_inv direct ---
    S_inv_mat_direct = jnp.linalg.inv(S_mat)

    # --- S_inv via Lyapunov ---
    def S_inv_vec_lyap(x):
        return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

    S_inv_mat_lyap = build_operator_matrix(S_inv_vec_lyap, dim, dtype)

    # --- Compose ---
    S_inv_K_lyap = S_inv_mat_lyap @ K_mat
    S_inv_K_direct = S_inv_mat_direct @ K_mat
    K_S_inv_lyap = K_mat @ S_inv_mat_lyap
    K_S_inv_direct = K_mat @ S_inv_mat_direct

    # --- Eigendecompositions ---
    evals_sinvk_lyap, evecs_sinvk_lyap = jnp.linalg.eig(S_inv_K_lyap)
    evals_sinvk_direct, evecs_sinvk_direct = jnp.linalg.eig(S_inv_K_direct)
    evals_ksinv_lyap, evecs_ksinv_lyap = jnp.linalg.eig(K_S_inv_lyap)
    evals_ksinv_direct, evecs_ksinv_direct = jnp.linalg.eig(K_S_inv_direct)

    # --- Dominant eigenvalue + eigenvector ---
    eval_sinvk_lyap, evec_sinvk_lyap = get_dominant_eigen(evals_sinvk_lyap, evecs_sinvk_lyap)
    eval_sinvk_direct, evec_sinvk_direct = get_dominant_eigen(evals_sinvk_direct, evecs_sinvk_direct)
    eval_ksinv_lyap, evec_ksinv_lyap = get_dominant_eigen(evals_ksinv_lyap, evecs_ksinv_lyap)
    eval_ksinv_direct, evec_ksinv_direct = get_dominant_eigen(evals_ksinv_direct, evecs_ksinv_direct)

    print(f"  S_inv @ K  — eval lyap: {eval_sinvk_lyap:.6f}, eval direct: {eval_sinvk_direct:.6f}")
    print(f"               overlap: {overlap(evec_sinvk_lyap, evec_sinvk_direct):.10f}")
    print(f"  K @ S_inv  — eval lyap: {eval_ksinv_lyap:.6f}, eval direct: {eval_ksinv_direct:.6f}")
    print(f"               overlap: {overlap(evec_ksinv_lyap, evec_ksinv_direct):.10f}")

    # --- Norm inf ---
    norminf_sinvk_lyap.append(
        compute_norminf(evecs_sinvk_lyap, evals_sinvk_lyap, H, Ls, n, dims)
    )
    norminf_sinvk_direct.append(
        compute_norminf(evecs_sinvk_direct, evals_sinvk_direct, H, Ls, n, dims)
    )
    norminf_ksinv_lyap.append(
        compute_norminf(evecs_ksinv_lyap, evals_ksinv_lyap, H, Ls, n, dims)
    )
    norminf_ksinv_direct.append(
        compute_norminf(evecs_ksinv_direct, evals_ksinv_direct, H, Ls, n, dims)
    )

# --- Print ---
print("\nnorminf_sinvk_lyap:", norminf_sinvk_lyap)
print("norminf_sinvk_direct:", norminf_sinvk_direct)
print("norminf_ksinv_lyap:", norminf_ksinv_lyap)
print("norminf_ksinv_direct:", norminf_ksinv_direct)

[1/1] delta/(2pi) = 3.00
  S_inv @ K  — eval lyap: -1.000000+0.000000j, eval direct: -1.000000+0.000000j
               overlap: 1.0000000000
  K @ S_inv  — eval lyap: -1.000000-0.000000j, eval direct: -1.000000-0.000000j
               overlap: 1.0000000000

norminf_sinvk_lyap: [6.182504708399003e-11]
norminf_sinvk_direct: [7.350930813807945e-11]
norminf_ksinv_lyap: [1.3841471870032023]
norminf_ksinv_direct: [1.384147187003196]


In [2]:
# Après avoir extrait les vecteurs propres dominants :
diff = jnp.linalg.norm(evec_sinvk_lyap - evec_sinvk_direct)
# Attention à la phase globale :
diff_phase = jnp.linalg.norm(
    evec_sinvk_lyap - evec_sinvk_direct * jnp.vdot(evec_sinvk_lyap, evec_sinvk_direct) / jnp.abs(jnp.vdot(evec_sinvk_lyap, evec_sinvk_direct))
)
print(f"  ||v_lyap - v_direct||: {diff:.2e}")
print(f"  ||v_lyap - v_direct|| (phase-aligned): {diff_phase:.2e}")

# Et surtout, compare les deux rho :
rho_lyap = to_dm(evec_sinvk_lyap, n=n, dims=dims)
rho_direct = to_dm(evec_sinvk_direct, n=n, dims=dims)
print(f"  ||rho_lyap - rho_direct||_inf: {float(jnp.max(jnp.abs(rho_lyap.to_jax() - rho_direct.to_jax()))):.2e}")

  ||v_lyap - v_direct||: 2.43e-14
  ||v_lyap - v_direct|| (phase-aligned): 2.57e-14
  ||rho_lyap - rho_direct||_inf: 9.40e-15


In [ ]:
from systems import build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

jax.config.update("jax_enable_x64", True)
n = 40
delta = 3 * 2 * jnp.pi


def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


H, Ls = build_kerr_oscillator(n, delta)
dims = H.dims
dtype = H.to_jax().dtype
dim = n * n

Ls_q = dq.stack(Ls)
H_jax = H.to_jax()
LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
G = -1j * H_jax - 0.5 * LdagL

lyap = LyapunovSolverEig(G, n_refinement=3)

# K matrix
K_mat = jnp.zeros((dim, dim), dtype=dtype)
for L in Ls:
    K_mat = K_mat + dq.sprepost(L, L.dag()).to_jax()

# S matrix
G_qa = dq.asqarray(G, dims=dims)
S_mat = (dq.spre(G_qa) + dq.spost(G_qa.dag())).to_jax()

# S_inv direct
S_inv_mat_direct = jnp.linalg.inv(S_mat)

# S_inv via Lyapunov
def S_inv_vec_lyap(x):
    return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

S_inv_mat_lyap = build_operator_matrix(S_inv_vec_lyap, dim, dtype)

# ============================================================
# TEST 1 : Comparer les matrices S_inv elles-mêmes
# ============================================================
diff_Sinv = jnp.max(jnp.abs(S_inv_mat_lyap - S_inv_mat_direct))
rel_diff_Sinv = diff_Sinv / jnp.max(jnp.abs(S_inv_mat_direct))
print("=== TEST 1: Comparaison des S_inv ===")
print(f"  ||S_inv_lyap - S_inv_direct||_inf: {float(diff_Sinv):.2e}")
print(f"  relative: {float(rel_diff_Sinv):.2e}")

# ============================================================
# TEST 2 : Vérifier que S_inv est bien l'inverse de S
# ============================================================
residual_lyap = jnp.max(jnp.abs(S_mat @ S_inv_mat_lyap - jnp.eye(dim, dtype=dtype)))
residual_direct = jnp.max(jnp.abs(S_mat @ S_inv_mat_direct - jnp.eye(dim, dtype=dtype)))
print("\n=== TEST 2: Résidu S @ S_inv - I ===")
print(f"  ||S @ S_inv_lyap - I||_inf: {float(residual_lyap):.2e}")
print(f"  ||S @ S_inv_direct - I||_inf: {float(residual_direct):.2e}")

# ============================================================
# TEST 3 : Comparer les matrices S_inv @ K
# ============================================================
S_inv_K_lyap =  K_mat@S_inv_mat_lyap 
S_inv_K_direct = K_mat@ S_inv_mat_direct 
diff_SinvK = jnp.max(jnp.abs(S_inv_K_lyap - S_inv_K_direct))
rel_diff_SinvK = diff_SinvK / jnp.max(jnp.abs(S_inv_K_direct))
print("\n=== TEST 3: Comparaison S_inv @ K ===")
print(f"  ||S_inv_K_lyap - S_inv_K_direct||_inf: {float(diff_SinvK):.2e}")
print(f"  relative: {float(rel_diff_SinvK):.2e}")

# ============================================================
# TEST 4 : Eigendecomposition et comparaison
# ============================================================
evals_lyap, evecs_lyap = jnp.linalg.eig(S_inv_K_lyap)
evals_direct, evecs_direct = jnp.linalg.eig(S_inv_K_direct)

idx_lyap = jnp.argsort(-jnp.abs(evals_lyap))[0]
idx_direct = jnp.argsort(-jnp.abs(evals_direct))[0]

eval_lyap = evals_lyap[idx_lyap]
eval_direct = evals_direct[idx_direct]
evec_lyap = evecs_lyap[:, idx_lyap]
evec_direct = evecs_direct[:, idx_direct]

print("\n=== TEST 4: Valeurs propres dominantes ===")
print(f"  eval_lyap:   {eval_lyap}")
print(f"  eval_direct: {eval_direct}")
print(f"  |eval_lyap - eval_direct|: {float(jnp.abs(eval_lyap - eval_direct)):.2e}")

# ============================================================
# TEST 5 : Résidu de l'equation aux vp : ||Mv - λv||
# ============================================================
res_lyap = jnp.linalg.norm(S_inv_K_lyap @ evec_lyap - eval_lyap * evec_lyap)
res_direct = jnp.linalg.norm(S_inv_K_direct @ evec_direct - eval_direct * evec_direct)
# Aussi : appliquer la matrice de l'AUTRE méthode
res_cross_lyap = jnp.linalg.norm(S_inv_K_direct @ evec_lyap - eval_lyap * evec_lyap)
res_cross_direct = jnp.linalg.norm(S_inv_K_lyap @ evec_direct - eval_direct * evec_direct)
print("\n=== TEST 5: Résidu ||Mv - λv|| ===")
print(f"  S_inv_K_lyap @ v_lyap - λ v_lyap:     {float(res_lyap):.2e}")
print(f"  S_inv_K_direct @ v_direct - λ v_direct: {float(res_direct):.2e}")
print(f"  S_inv_K_direct @ v_lyap - λ v_lyap:    {float(res_cross_lyap):.2e}")
print(f"  S_inv_K_lyap @ v_direct - λ v_direct:  {float(res_cross_direct):.2e}")

# ============================================================
# TEST 6 : Comparer rho et norminf en détail
# ============================================================
rho_lyap = to_dm(evec_lyap, n=n, dims=dims)
rho_direct = to_dm(evec_direct, n=n, dims=dims)

Lrho_lyap = dq.lindbladian(H, Ls, rho_lyap).to_jax()
Lrho_direct = dq.lindbladian(H, Ls, rho_direct).to_jax()

print("\n=== TEST 6: norminf de L(rho) ===")
print(f"  ||L(rho_lyap)||_inf:   {float(jnp.max(jnp.abs(Lrho_lyap))):.2e}")
print(f"  ||L(rho_direct)||_inf: {float(jnp.max(jnp.abs(Lrho_direct))):.2e}")
print(f"  ||rho_lyap - rho_direct||_inf: {float(jnp.max(jnp.abs(rho_lyap.to_jax() - rho_direct.to_jax()))):.2e}")

# ============================================================
# TEST 7 : Est-ce que le même vecteur propre donne des norminf
#           différents selon la matrice S_inv_K utilisée ?
#           (utiliser evec_direct avec les DEUX matrices)
# ============================================================
rho_from_direct = to_dm(evec_direct, n=n, dims=dims)
Lrho_from_direct = dq.lindbladian(H, Ls, rho_from_direct).to_jax()
print("\n=== TEST 7: Même vecteur, même rho, même norminf ? ===")
print(f"  evec_direct -> norminf: {float(jnp.max(jnp.abs(Lrho_from_direct))):.2e}")

# Vérification : est-ce que to_dm est linéaire / déterministe ?
rho_check1 = to_dm(evec_direct, n=n, dims=dims)
rho_check2 = to_dm(evec_direct, n=n, dims=dims)
print(f"  to_dm deterministic? ||rho1 - rho2||: {float(jnp.max(jnp.abs(rho_check1.to_jax() - rho_check2.to_jax()))):.2e}")

# ============================================================
# TEST 8 : Vérifier la normalisation du vecteur propre
# ============================================================
print("\n=== TEST 8: Normalisation ===")
print(f"  ||evec_lyap||:   {float(jnp.linalg.norm(evec_lyap)):.15f}")
print(f"  ||evec_direct||: {float(jnp.linalg.norm(evec_direct)):.15f}")
print(f"  tr(rho_lyap):    {dq.trace(rho_lyap)}")
print(f"  tr(rho_direct):  {dq.trace(rho_direct)}")

=== TEST 1: Comparaison des S_inv ===
  ||S_inv_lyap - S_inv_direct||_inf: 3.49e-18
  relative: 5.58e-16

=== TEST 2: Résidu S @ S_inv - I ===
  ||S @ S_inv_lyap - I||_inf: 4.58e-16
  ||S @ S_inv_direct - I||_inf: 8.88e-16

=== TEST 3: Comparaison S_inv @ K ===
  ||S_inv_K_lyap - S_inv_K_direct||_inf: 6.66e-16
  relative: 8.03e-16

=== TEST 4: Valeurs propres dominantes ===
  eval_lyap:   (-1.0000000000000098+3.0531133177191805e-16j)
  eval_direct: (-1.0000000000000033+2.862293735361732e-17j)
  |eval_lyap - eval_direct|: 6.45e-15

=== TEST 5: Résidu ||Mv - λv|| ===
  S_inv_K_lyap @ v_lyap - λ v_lyap:     3.11e-14
  S_inv_K_direct @ v_direct - λ v_direct: 2.84e-14
  S_inv_K_direct @ v_lyap - λ v_lyap:    3.11e-14
  S_inv_K_lyap @ v_direct - λ v_direct:  2.84e-14

=== TEST 6: norminf de L(rho) ===
  ||L(rho_lyap)||_inf:   6.18e-11
  ||L(rho_direct)||_inf: 7.35e-11
  ||rho_lyap - rho_direct||_inf: 9.40e-15

=== TEST 7: Même vecteur, même rho, même norminf ? ===
  evec_direct -> norminf: 7

In [4]:
from systems import build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

jax.config.update("jax_enable_x64", True)
n = 40
delta = 3 * 2 * jnp.pi


def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


H, Ls = build_kerr_oscillator(n, delta)
dims = H.dims
dtype = H.to_jax().dtype
dim = n * n

Ls_q = dq.stack(Ls)
H_jax = H.to_jax()
LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
G = -1j * H_jax - 0.5 * LdagL

lyap = LyapunovSolverEig(G, n_refinement=3)

# K matrix
K_mat = jnp.zeros((dim, dim), dtype=dtype)
for L in Ls:
    K_mat = K_mat + dq.sprepost(L, L.dag()).to_jax()

# S matrix
G_qa = dq.asqarray(G, dims=dims)
S_mat = (dq.spre(G_qa) + dq.spost(G_qa.dag())).to_jax()

# S_inv direct
S_inv_mat_direct = jnp.linalg.inv(S_mat)

# S_inv via Lyapunov
def S_inv_vec_lyap(x):
    return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

S_inv_mat_lyap = build_operator_matrix(S_inv_vec_lyap, dim, dtype)

# ============================================================
# TEST 1 : Comparer les matrices S_inv elles-mêmes
# ============================================================
diff_Sinv = jnp.max(jnp.abs(S_inv_mat_lyap - S_inv_mat_direct))
rel_diff_Sinv = diff_Sinv / jnp.max(jnp.abs(S_inv_mat_direct))
print("=== TEST 1: Comparaison des S_inv ===")
print(f"  ||S_inv_lyap - S_inv_direct||_inf: {float(diff_Sinv):.2e}")
print(f"  relative: {float(rel_diff_Sinv):.2e}")

# ============================================================
# TEST 2 : Vérifier que S_inv est bien l'inverse de S
# ============================================================
residual_lyap = jnp.max(jnp.abs(S_mat @ S_inv_mat_lyap - jnp.eye(dim, dtype=dtype)))
residual_direct = jnp.max(jnp.abs(S_mat @ S_inv_mat_direct - jnp.eye(dim, dtype=dtype)))
print("\n=== TEST 2: Résidu S @ S_inv - I ===")
print(f"  ||S @ S_inv_lyap - I||_inf: {float(residual_lyap):.2e}")
print(f"  ||S @ S_inv_direct - I||_inf: {float(residual_direct):.2e}")

# ============================================================
# TEST 3 : Comparer les matrices S_inv @ K
# ============================================================
S_inv_K_lyap =  K_mat@S_inv_mat_lyap 
S_inv_K_direct = K_mat@ S_inv_mat_direct 
diff_SinvK = jnp.max(jnp.abs(S_inv_K_lyap - S_inv_K_direct))
rel_diff_SinvK = diff_SinvK / jnp.max(jnp.abs(S_inv_K_direct))
print("\n=== TEST 3: Comparaison S_inv @ K ===")
print(f"  ||S_inv_K_lyap - S_inv_K_direct||_inf: {float(diff_SinvK):.2e}")
print(f"  relative: {float(rel_diff_SinvK):.2e}")

# ============================================================
# TEST 4 : Eigendecomposition et comparaison
# ============================================================
evals_lyap, evecs_lyap = jnp.linalg.eig(S_inv_K_lyap)
evals_direct, evecs_direct = jnp.linalg.eig(S_inv_K_direct)

idx_lyap = jnp.argsort(-jnp.abs(evals_lyap))[0]
idx_direct = jnp.argsort(-jnp.abs(evals_direct))[0]

eval_lyap = evals_lyap[idx_lyap]
eval_direct = evals_direct[idx_direct]
evec_lyap = evecs_lyap[:, idx_lyap]
evec_direct = evecs_direct[:, idx_direct]

print("\n=== TEST 4: Valeurs propres dominantes ===")
print(f"  eval_lyap:   {eval_lyap}")
print(f"  eval_direct: {eval_direct}")
print(f"  |eval_lyap - eval_direct|: {float(jnp.abs(eval_lyap - eval_direct)):.2e}")

# ============================================================
# TEST 5 : Résidu de l'equation aux vp : ||Mv - λv||
# ============================================================
res_lyap = jnp.linalg.norm(S_inv_K_lyap @ evec_lyap - eval_lyap * evec_lyap)
res_direct = jnp.linalg.norm(S_inv_K_direct @ evec_direct - eval_direct * evec_direct)
# Aussi : appliquer la matrice de l'AUTRE méthode
res_cross_lyap = jnp.linalg.norm(S_inv_K_direct @ evec_lyap - eval_lyap * evec_lyap)
res_cross_direct = jnp.linalg.norm(S_inv_K_lyap @ evec_direct - eval_direct * evec_direct)
print("\n=== TEST 5: Résidu ||Mv - λv|| ===")
print(f"  S_inv_K_lyap @ v_lyap - λ v_lyap:     {float(res_lyap):.2e}")
print(f"  S_inv_K_direct @ v_direct - λ v_direct: {float(res_direct):.2e}")
print(f"  S_inv_K_direct @ v_lyap - λ v_lyap:    {float(res_cross_lyap):.2e}")
print(f"  S_inv_K_lyap @ v_direct - λ v_direct:  {float(res_cross_direct):.2e}")

# ============================================================
# TEST 6 : Comparer rho et norminf en détail
# ============================================================
rho_lyap = to_dm(evec_lyap, n=n, dims=dims)
rho_direct = to_dm(evec_direct, n=n, dims=dims)

Lrho_lyap = dq.lindbladian(H, Ls, rho_lyap).to_jax()
Lrho_direct = dq.lindbladian(H, Ls, rho_direct).to_jax()

print("\n=== TEST 6: norminf de L(rho) ===")
print(f"  ||L(rho_lyap)||_inf:   {float(jnp.max(jnp.abs(Lrho_lyap))):.2e}")
print(f"  ||L(rho_direct)||_inf: {float(jnp.max(jnp.abs(Lrho_direct))):.2e}")
print(f"  ||rho_lyap - rho_direct||_inf: {float(jnp.max(jnp.abs(rho_lyap.to_jax() - rho_direct.to_jax()))):.2e}")

# ============================================================
# TEST 7 : Est-ce que le même vecteur propre donne des norminf
#           différents selon la matrice S_inv_K utilisée ?
#           (utiliser evec_direct avec les DEUX matrices)
# ============================================================
rho_from_direct = to_dm(evec_direct, n=n, dims=dims)
Lrho_from_direct = dq.lindbladian(H, Ls, rho_from_direct).to_jax()
print("\n=== TEST 7: Même vecteur, même rho, même norminf ? ===")
print(f"  evec_direct -> norminf: {float(jnp.max(jnp.abs(Lrho_from_direct))):.2e}")

# Vérification : est-ce que to_dm est linéaire / déterministe ?
rho_check1 = to_dm(evec_direct, n=n, dims=dims)
rho_check2 = to_dm(evec_direct, n=n, dims=dims)
print(f"  to_dm deterministic? ||rho1 - rho2||: {float(jnp.max(jnp.abs(rho_check1.to_jax() - rho_check2.to_jax()))):.2e}")

# ============================================================
# TEST 8 : Vérifier la normalisation du vecteur propre
# ============================================================
print("\n=== TEST 8: Normalisation ===")
print(f"  ||evec_lyap||:   {float(jnp.linalg.norm(evec_lyap)):.15f}")
print(f"  ||evec_direct||: {float(jnp.linalg.norm(evec_direct)):.15f}")
print(f"  tr(rho_lyap):    {dq.trace(rho_lyap)}")
print(f"  tr(rho_direct):  {dq.trace(rho_direct)}")

=== TEST 1: Comparaison des S_inv ===
  ||S_inv_lyap - S_inv_direct||_inf: 3.49e-18
  relative: 5.58e-16

=== TEST 2: Résidu S @ S_inv - I ===
  ||S @ S_inv_lyap - I||_inf: 4.58e-16
  ||S @ S_inv_direct - I||_inf: 8.88e-16

=== TEST 3: Comparaison S_inv @ K ===
  ||S_inv_K_lyap - S_inv_K_direct||_inf: 4.58e-16
  relative: 5.16e-16

=== TEST 4: Valeurs propres dominantes ===
  eval_lyap:   (-0.9999999999999913-1.6653345369377348e-16j)
  eval_direct: (-0.9999999999999958-6.938893903907228e-17j)
  |eval_lyap - eval_direct|: 4.44e-15

=== TEST 5: Résidu ||Mv - λv|| ===
  S_inv_K_lyap @ v_lyap - λ v_lyap:     1.07e-14
  S_inv_K_direct @ v_direct - λ v_direct: 7.01e-15
  S_inv_K_direct @ v_lyap - λ v_lyap:    1.07e-14
  S_inv_K_lyap @ v_direct - λ v_direct:  7.04e-15

=== TEST 6: norminf de L(rho) ===
  ||L(rho_lyap)||_inf:   1.38e+00
  ||L(rho_direct)||_inf: 1.38e+00
  ||rho_lyap - rho_direct||_inf: 2.02e-15

=== TEST 7: Même vecteur, même rho, même norminf ? ===
  evec_direct -> norminf: 1

In [5]:
from systems import build_kerr_oscillator
import dynamiqs as dq
import jax
import jax.numpy as jnp

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

jax.config.update("jax_enable_x64", True)
n = 40
delta = 3 * 2 * jnp.pi


def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


H, Ls = build_kerr_oscillator(n, delta)
dims = H.dims
dtype = H.to_jax().dtype
dim = n * n

Ls_q = dq.stack(Ls)
H_jax = H.to_jax()
LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
G = -1j * H_jax - 0.5 * LdagL

lyap = LyapunovSolverEig(G, n_refinement=3)

# K matrix
K_mat = jnp.zeros((dim, dim), dtype=dtype)
for L in Ls:
    K_mat = K_mat + dq.sprepost(L, L.dag()).to_jax()

# S matrix
G_qa = dq.asqarray(G, dims=dims)
S_mat = (dq.spre(G_qa) + dq.spost(G_qa.dag())).to_jax()

# S_inv (les deux méthodes)
S_inv_direct = jnp.linalg.inv(S_mat)

def S_inv_vec_lyap(x):
    return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

S_inv_lyap = build_operator_matrix(S_inv_vec_lyap, dim, dtype)

# Les 4 matrices
S_inv_K_lyap = S_inv_lyap @ K_mat
S_inv_K_direct = S_inv_direct @ K_mat
K_S_inv_lyap = K_mat @ S_inv_lyap
K_S_inv_direct = K_mat @ S_inv_direct

matrices = {
    "S_inv_K (lyap)":   S_inv_K_lyap,
    "S_inv_K (direct)": S_inv_K_direct,
    "K_S_inv (lyap)":   K_S_inv_lyap,
    "K_S_inv (direct)": K_S_inv_direct,
}

for name, M in matrices.items():
    evals, evecs = jnp.linalg.eig(M)
    idx = jnp.argsort(-jnp.abs(evals))[0]
    lam = evals[idx]
    v = evecs[:, idx]

    rho = to_dm(v, n=n, dims=dims)
    Lrho = dq.lindbladian(H, Ls, rho).to_jax()
    norminf = float(jnp.max(jnp.abs(Lrho)))
    tr = dq.trace(rho)

    print(f"{name}:")
    print(f"  lambda = {lam}")
    print(f"  |lambda| = {float(jnp.abs(lam)):.10f}")
    print(f"  tr(rho) = {tr}")
    print(f"  ||L(rho)||_inf = {norminf:.2e}")
    print()

S_inv_K (lyap):
  lambda = (-1.0000000000000098+3.0531133177191805e-16j)
  |lambda| = 1.0000000000
  tr(rho) = (1.0941091204970297+3.1498786197750962e-15j)
  ||L(rho)||_inf = 6.18e-11

S_inv_K (direct):
  lambda = (-1.0000000000000033+2.862293735361732e-17j)
  |lambda| = 1.0000000000
  tr(rho) = (1.0941091204970312+4.0146984613298153e-16j)
  ||L(rho)||_inf = 7.35e-11

K_S_inv (lyap):
  lambda = (-0.9999999999999913-1.6653345369377348e-16j)
  |lambda| = 1.0000000000
  tr(rho) = (1.0867312486462675-1.377838070541029e-15j)
  ||L(rho)||_inf = 1.38e+00

K_S_inv (direct):
  lambda = (-0.9999999999999958-6.938893903907228e-17j)
  |lambda| = 1.0000000000
  tr(rho) = (1.0867312486462615-5.650436222241432e-16j)
  ||L(rho)||_inf = 1.38e+00



In [6]:
from systems import build_kerr_oscillator, build_random_single_mode
import dynamiqs as dq
import jax
import jax.numpy as jnp

from dynamiqs.steady_state.api.utils import from_matrix, to_dm, to_matrix
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

jax.config.update("jax_enable_x64", True)
n = 40
delta = 3 * 2 * jnp.pi



def build_operator_matrix(apply_fn, dim, dtype):
    M = jnp.zeros((dim, dim), dtype=dtype)
    for j in range(dim):
        e_j = jnp.zeros(dim, dtype=dtype).at[j].set(1.0)
        M = M.at[:, j].set(apply_fn(e_j))
    return M


H, Ls = build_random_single_mode(n, gamma=1.0)
dims = H.dims
dtype = H.to_jax().dtype
dim = n * n

Ls_q = dq.stack(Ls)
H_jax = H.to_jax()
LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
G = -1j * H_jax - 0.5 * LdagL

lyap = LyapunovSolverEig(G, n_refinement=3)

# K matrix
K_mat = jnp.zeros((dim, dim), dtype=dtype)
for L in Ls:
    K_mat = K_mat + dq.sprepost(L, L.dag()).to_jax()

# S matrix
G_qa = dq.asqarray(G, dims=dims)
S_mat = (dq.spre(G_qa) + dq.spost(G_qa.dag())).to_jax()

# S_inv (les deux méthodes)
S_inv_direct = jnp.linalg.inv(S_mat)

def S_inv_vec_lyap(x):
    return from_matrix(lyap.solve(to_matrix(x, n=n), mu=0.0))

S_inv_lyap = build_operator_matrix(S_inv_vec_lyap, dim, dtype)

# Les 4 matrices
S_inv_K_lyap = S_inv_lyap @ K_mat
S_inv_K_direct = S_inv_direct @ K_mat
K_S_inv_lyap = K_mat @ S_inv_lyap
K_S_inv_direct = K_mat @ S_inv_direct

matrices = {
    "S_inv_K (lyap)":   S_inv_K_lyap,
    "S_inv_K (direct)": S_inv_K_direct,
    "K_S_inv (lyap)":   K_S_inv_lyap,
    "K_S_inv (direct)": K_S_inv_direct,
}

for name, M in matrices.items():
    evals, evecs = jnp.linalg.eig(M)
    idx = jnp.argsort(-jnp.abs(evals))[0]
    lam = evals[idx]
    v = evecs[:, idx]

    rho = to_dm(v, n=n, dims=dims)
    Lrho = dq.lindbladian(H, Ls, rho).to_jax()
    norminf = float(jnp.max(jnp.abs(Lrho)))
    tr = dq.trace(rho)

    print(f"{name}:")
    print(f"  lambda = {lam}")
    print(f"  |lambda| = {float(jnp.abs(lam)):.10f}")
    print(f"  tr(rho) = {tr}")
    print(f"  ||L(rho)||_inf = {norminf:.2e}")
    print()

S_inv_K (lyap):
  lambda = (-0.9999999999999998+0j)
  |lambda| = 1.0000000000
  tr(rho) = (4.739880424957166-1.2917662544920098e-15j)
  ||L(rho)||_inf = 1.44e-13

S_inv_K (direct):
  lambda = (-0.999999999999998-1.5959455978986625e-16j)
  |lambda| = 1.0000000000
  tr(rho) = (4.7398804249571675-3.8335897811220945e-17j)
  ||L(rho)||_inf = 1.31e-13

K_S_inv (lyap):
  lambda = (-0.9999999999999998+8.326672684688674e-17j)
  |lambda| = 1.0000000000
  tr(rho) = (5.476214880239742-8.456822384658603e-16j)
  ||L(rho)||_inf = 1.28e+01

K_S_inv (direct):
  lambda = (-0.9999999999999978+1.3877787807814062e-17j)
  |lambda| = 1.0000000000
  tr(rho) = (5.476214880239743+6.709899643558717e-16j)
  ||L(rho)||_inf = 1.28e+01



In [7]:
# Eigenvecs dominants
evals_sinvk, evecs_sinvk = jnp.linalg.eig(S_inv_K_direct)
evals_ksinv, evecs_ksinv = jnp.linalg.eig(K_S_inv_direct)

idx_sinvk = jnp.argsort(-jnp.abs(evals_sinvk))[0]
idx_ksinv = jnp.argsort(-jnp.abs(evals_ksinv))[0]

v_sinvk = evecs_sinvk[:, idx_sinvk]
v_ksinv = evecs_ksinv[:, idx_ksinv]

# Overlap direct
ov = jnp.abs(jnp.vdot(v_sinvk, v_ksinv)) / (jnp.linalg.norm(v_sinvk) * jnp.linalg.norm(v_ksinv))
print(f"overlap(v_sinvk, v_ksinv) = {float(ov):.15f}")

# Comparer les rho
rho_sinvk = to_dm(v_sinvk, n=n, dims=dims)
rho_ksinv = to_dm(v_ksinv, n=n, dims=dims)
print(f"||rho_sinvk - rho_ksinv||_inf = {float(jnp.max(jnp.abs(rho_sinvk.to_jax() - rho_ksinv.to_jax()))):.2e}")

# Vérifier la relation théorique : v_ksinv ~ K @ v_sinvk ?
Kv = K_mat @ v_sinvk
ov_theory = jnp.abs(jnp.vdot(Kv, v_ksinv)) / (jnp.linalg.norm(Kv) * jnp.linalg.norm(v_ksinv))
print(f"overlap(K @ v_sinvk, v_ksinv) = {float(ov_theory):.15f}")

overlap(v_sinvk, v_ksinv) = 0.816395163300364
||rho_sinvk - rho_ksinv||_inf = 4.91e-02
overlap(K @ v_sinvk, v_ksinv) = 0.999999999999998
